Слепой набор протоколов: 60 стратифицированных кейсов x 4 системы.....


In [1]:
from pathlib import Path
import os
_REPOSITORY_ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()))
if os.name == 'nt' and (not str(_REPOSITORY_ROOT).startswith('\\\\?\\')):
    _REPOSITORY_ROOT = Path('\\\\?\\' + str(_REPOSITORY_ROOT))
_NOTEBOOK_DIR = _REPOSITORY_ROOT / 'evaluations/human_and_model_validation'
os.chdir(_NOTEBOOK_DIR)


In [2]:
import hashlib
import os
import re
from typing import Dict, List, Optional, Tuple
import numpy as np
import pandas as pd

def _read_csv(*args, **kwargs):
    frame = pd.read_csv(*args, **kwargs)
    for column in frame.select_dtypes(include=['object', 'string']).columns:
        frame[column] = frame[column].map(lambda value: value.replace('\r\n', '\n').replace('\r', '\n') if isinstance(value, str) else value)
    return frame
HERE = os.path.abspath(os.getcwd())
GOLD = str(_REPOSITORY_ROOT)
NOFIX = os.path.join(GOLD, 'experiments/clinical_protocols')
DELPHI_DIR = 'heterogeneous_gpt120_organizer_complete'
DELPHI_DIR_FAILED = 'heterogeneous_gpt120_organizer_truncated'
SOURCES = [('delphi', os.path.join(NOFIX, 'delphi_panels', DELPHI_DIR, 'delphi_noroles.csv'), 'winner_protocol'), ('deepseekv4', os.path.join(NOFIX, 'single_model_baselines', 'baseline_deepseekv4', 'baseline_noroles_deepseekv4.csv'), 'baseline_protocol'), ('gpt', os.path.join(NOFIX, 'single_model_baselines', 'baseline_gpt', 'baseline_noroles_gpt.csv'), 'baseline_protocol'), ('qwen', os.path.join(NOFIX, 'single_model_baselines', 'baseline_qwen', 'baseline_noroles_qwen.csv'), 'baseline_protocol')]
SOURCE_ORDER = [name for name, _, _ in SOURCES]
MERGE_KEY = 'Vignette'
N_CASES = 60
SELECTION_SEED = 20260831
PERMUTATION_SEED = 20260824
STRATA = {'Genealogy': ['AI', 'checker', 'msd'], 'Type': ['Requires emergent care', 'Requires non-emergent care', 'Self-care appropriate']}
STRATA_ROW, STRATA_COL = list(STRATA)
ALLOCATION_MODE = 'balanced'
MIN_CAP_HITS = 3
MIN_PROTOCOL_CHARS = 400
REQUIRED_SECTIONS = ['Age', 'Sex', 'Complaint', 'Analysis', 'Required diagnostic equipment', 'Conclusion', 'Emergency type', 'Subclinical specialization', 'Recommendations', 'Notes']
DELPHI_EXPERTS = ['deepseekv4_default', 'gpt120b_default', 'qwen_default']
CASE_COLUMNS = ['case_id', 'Vignette', 'case_text', 'Criterion', 'Diagnosis_vulgar', 'Type', 'Genealogy', 'Sub Clinical Specialties', 'Age Group']
OUT_BLIND = os.path.join(HERE, 'blind_protocols_60.csv')
OUT_KEYED = os.path.join(HERE, 'blind_protocols_60_key.csv')
OUT_SELECTION = os.path.join(HERE, 'blind_protocols_60_selection.csv')
pd.set_option('display.width', 200)
print('HERE :', HERE)
print('NOFIX:', NOFIX)
print('Источников:', len(SOURCES), '| порядок для perm_key:', SOURCE_ORDER)
print('Режим отбора:', ALLOCATION_MODE)
for name, path, _ in SOURCES:
    print(f"  {name:<12} {('OK' if os.path.exists(path) else 'НЕТ ФАЙЛА')}")


HERE : .\evaluations\human_and_model_validation
NOFIX: .\experiments/clinical_protocols
Источников: 4 | порядок для perm_key: ['delphi', 'deepseekv4', 'gpt', 'qwen']
Режим отбора: balanced
  delphi       НЕТ ФАЙЛА
  deepseekv4   НЕТ ФАЙЛА
  gpt          НЕТ ФАЙЛА
  qwen         НЕТ ФАЙЛА


In [3]:
def detect_cap(series: pd.Series) -> Tuple[Optional[int], int, int]:
    values = pd.to_numeric(series, errors='coerce').dropna()
    if values.empty:
        return (None, 0, 0)
    top = values.max()
    hits = int((values == top).sum())
    return (int(top) if hits >= MIN_CAP_HITS else None, hits, len(values))

def delphi_diagnostics(path: str, label: str) -> None:
    if not os.path.exists(path):
        print(f'{label}: файла нет, пропуск')
        return
    run = _read_csv(path, sep=';', encoding='utf-8-sig', low_memory=False)
    run = run[run['table_source'].astype(str) == 'result_df']
    run = run[run[MERGE_KEY].notna()].reset_index(drop=True)
    print(f'--- {label}  ({len(run)} кейсов)')
    capped_calls = 0
    for rnd in (1, 2, 3):
        cells = []
        for expert in DELPHI_EXPERTS:
            column = f'r{rnd}_completion_tokens__{expert}'
            if column not in run.columns:
                continue
            cap, hits, total = detect_cap(run[column])
            short = expert.split('_')[0]
            if cap is not None:
                capped_calls += hits
                cells.append(f'{short}: ОБРЕЗАН на {cap} ({hits}/{total})')
            else:
                top = int(pd.to_numeric(run[column], errors='coerce').max())
                cells.append(f'{short}: ок (max {top})')
        vote_columns = [f'r{rnd}_emergency__{e}' for e in DELPHI_EXPERTS if f'r{rnd}_emergency__{e}' in run.columns]
        votes = run[vote_columns].notna().sum(axis=1)
        played = int((votes > 0).sum())
        full = int((votes == len(vote_columns)).sum())
        print(f'  r{rnd}: ' + ' | '.join(cells))
        print(f'      раунд сыгран в {played} кейсах, панель полна (3/3 голоса) в {full}' + ('' if played == full else '  <-- НЕПОЛНЫЕ РАУНДЫ'))
    print(f'  обрезанных вызовов всего: {capped_calls}')
delphi_diagnostics(os.path.join(NOFIX, 'delphi_panels', DELPHI_DIR_FAILED, 'delphi_noroles.csv'), 'СТАРЫЙ прогон (зажатый лимит) - НЕ используется')
print()
delphi_diagnostics(os.path.join(NOFIX, 'delphi_panels', DELPHI_DIR, 'delphi_noroles.csv'), 'НОВЫЙ прогон (_full) - идёт в набор')


СТАРЫЙ прогон (зажатый лимит) - НЕ используется: файла нет, пропуск

НОВЫЙ прогон (_full) - идёт в набор: файла нет, пропуск


In [4]:
def case_id(vignette: str) -> str:
    return hashlib.sha1(str(vignette).strip().encode('utf-8')).hexdigest()[:10]

def read_run(path: str) -> pd.DataFrame:
    df = _read_csv(os.path.abspath(path), sep=';', encoding='utf-8-sig', low_memory=False)
    total = len(df)
    if 'table_source' in df.columns:
        service = df['table_source'].astype(str) != 'result_df'
    else:
        service = pd.Series(False, index=df.index)
    df = df[~service]
    df = df[df[MERGE_KEY].notna() & (df[MERGE_KEY].astype(str).str.strip() != '')]
    df = df.reset_index(drop=True).copy()
    df[MERGE_KEY] = df[MERGE_KEY].astype(str)
    df['case_id'] = df[MERGE_KEY].map(case_id)
    print(f'  {os.path.basename(os.path.dirname(path))[:52]:<52} {total} строк -> {len(df)} кейсов (служебных отброшено: {total - len(df)})')
    return df
print('Чтение прогонов:')
runs: Dict[str, pd.DataFrame] = {}
protocol_column: Dict[str, str] = {}
for name, path, column in SOURCES:
    runs[name] = read_run(path)
    protocol_column[name] = column
ordered_keys = list(runs[SOURCE_ORDER[0]][MERGE_KEY])
print()
print('Выравнивание источников (порядок кейсов совпадает с дельфийским):')
for name in SOURCE_ORDER:
    print(f'  {name:<12} {list(runs[name][MERGE_KEY]) == ordered_keys}')
corpus = runs[SOURCE_ORDER[0]].set_index(MERGE_KEY)
print()
print(f'Корпус: {len(ordered_keys)} кейсов')
for column, levels in STRATA.items():
    counts = corpus[column].value_counts()
    print(f'  {column:<10} ' + ', '.join((f'{lv}: {int(counts.get(lv, 0))}' for lv in levels)))


Чтение прогонов:


  heterogeneous_gpt120_organizer_complete              235 строк -> 229 кейсов (служебных отброшено: 6)
  baseline_deepseekv4                                  230 строк -> 229 кейсов (служебных отброшено: 1)


  baseline_gpt                                         230 строк -> 229 кейсов (служебных отброшено: 1)
  baseline_qwen                                        230 строк -> 229 кейсов (служебных отброшено: 1)

Выравнивание источников (порядок кейсов совпадает с дельфийским):
  delphi       True
  deepseekv4   True
  gpt          True
  qwen         True

Корпус: 229 кейсов
  Genealogy  AI: 151, checker: 43, msd: 35
  Type       Requires emergent care: 84, Requires non-emergent care: 81, Self-care appropriate: 64


In [5]:
SECTION_PATTERNS = {section: re.compile(f'(?im)^[\\s#*\\->]*\\**\\s*{re.escape(section)}') for section in REQUIRED_SECTIONS}

def strip_author_section(protocol_text: Optional[str]) -> str:
    if protocol_text is None:
        return ''
    text = str(protocol_text).strip()
    parts = re.split('(?im)^\\s*Protocol Author\\s*:', text, maxsplit=1)
    return parts[0].rstrip()

def completeness(name: str) -> pd.DataFrame:
    run = runs[name]
    text = run[protocol_column[name]].map(strip_author_section)
    reasons = pd.Series('', index=run.index)
    too_short = text.str.len() < MIN_PROTOCOL_CHARS
    reasons[too_short & (reasons == '')] = 'пустой/огрызок'
    missing = pd.Series([[] for _ in run.index], index=run.index)
    for section, pattern in SECTION_PATTERNS.items():
        for i in run.index[~text.str.contains(pattern)]:
            missing[i] = missing[i] + [section]
    lacks = (missing.map(len) > 0) & (reasons == '')
    reasons[lacks] = missing[lacks].map(lambda s: 'нет секций: ' + ', '.join(s))
    token_column = 'final_synthesis_completion_tokens' if name == 'delphi' else 'completion_tokens'
    if token_column in run.columns:
        cap, hits, total = detect_cap(run[token_column])
        if cap is not None:
            at_cap = pd.to_numeric(run[token_column], errors='coerce') == cap
            reasons[at_cap & (reasons == '')] = f'финальная генерация упёрлась в лимит {cap}'
            print(f'  {name:<12} лимит финальной генерации {cap}, на нём {hits}/{total} вызовов')
    if name == 'delphi':
        for rnd in (1, 2, 3):
            columns = [f'r{rnd}_emergency__{e}' for e in DELPHI_EXPERTS if f'r{rnd}_emergency__{e}' in run.columns]
            if not columns:
                continue
            votes = run[columns].notna().sum(axis=1)
            broken = (votes > 0) & (votes < len(columns))
            reasons[broken & (reasons == '')] = f'r{rnd}: неполная панель'
            for expert in DELPHI_EXPERTS:
                column = f'r{rnd}_completion_tokens__{expert}'
                if column not in run.columns:
                    continue
                cap, _, _ = detect_cap(run[column])
                if cap is None:
                    continue
                at_cap = pd.to_numeric(run[column], errors='coerce') == cap
                reasons[at_cap & (reasons == '')] = f'r{rnd}/{expert}: обрезан на {cap}'
    return pd.DataFrame({MERGE_KEY: run[MERGE_KEY], 'case_id': run['case_id'], 'text': text, 'ok': reasons == '', 'reason': reasons}).set_index(MERGE_KEY)
print('Полнота протоколов по источникам:')
complete_tables = {name: completeness(name) for name in SOURCE_ORDER}
print()
for name in SOURCE_ORDER:
    table = complete_tables[name]
    bad = table[~table['ok']]
    print(f"  {name:<12} полных {int(table['ok'].sum())}/{len(table)}" + (f'  | выпало {len(bad)}' if len(bad) else ''))
    for reason, group in bad.groupby('reason'):
        ids = ', '.join(group['case_id'].head(6)) + (', …' if len(group) > 6 else '')
        print(f'      {reason}: {len(group)} ({ids})')
eligible_mask = pd.Series(True, index=pd.Index(ordered_keys, name=MERGE_KEY))
for name in SOURCE_ORDER:
    eligible_mask &= complete_tables[name]['ok'].reindex(eligible_mask.index).fillna(False)
eligible = list(eligible_mask[eligible_mask].index)
print()
print(f'ПУЛ: кейсов, где полны все {len(SOURCE_ORDER)} протокола: {len(eligible)} из {len(ordered_keys)}')
pool = corpus.loc[eligible]
pool_table = pd.crosstab(pool[STRATA_ROW], pool[STRATA_COL]).reindex(index=STRATA[STRATA_ROW], columns=STRATA[STRATA_COL], fill_value=0)
print()
print('Пул в разрезе страт:')
print(pool_table.to_string())


Полнота протоколов по источникам:



  delphi       полных 226/229  | выпало 3
      r1: неполная панель: 2 (b638d9e0be, ba1a64be3b)
      r2: неполная панель: 1 (05611d537b)
  deepseekv4   полных 226/229  | выпало 3
      нет секций: Required diagnostic equipment: 2 (810555234b, e4716c9085)
      пустой/огрызок: 1 (9b0f33ad8d)
  gpt          полных 229/229
  qwen         полных 229/229

ПУЛ: кейсов, где полны все 4 протокола: 223 из 229



Пул в разрезе страт:
Type       Requires emergent care  Requires non-emergent care  Self-care appropriate
Genealogy                                                                           
AI                             52                          46                     51
checker                        14                          16                     13
msd                            15                          16                      0


In [6]:
def ipf(seed_table: np.ndarray, row_targets: np.ndarray, col_targets: np.ndarray, iterations: int=5000) -> np.ndarray:
    matrix = seed_table.astype(float).copy()
    for _ in range(iterations):
        rows = matrix.sum(axis=1)
        matrix *= np.divide(row_targets, rows, out=np.zeros_like(rows), where=rows > 0)[:, None]
        cols = matrix.sum(axis=0)
        matrix *= np.divide(col_targets, cols, out=np.zeros_like(cols), where=cols > 0)[None, :]
    return matrix

def round_preserving_margins(matrix: np.ndarray, row_targets: np.ndarray, col_targets: np.ndarray, caps: np.ndarray) -> np.ndarray:
    floor = np.floor(matrix + 1e-09).astype(int)
    remainder = matrix - floor
    need_rows = np.rint(row_targets).astype(int) - floor.sum(axis=1)
    need_cols = np.rint(col_targets).astype(int) - floor.sum(axis=0)
    order = sorted(((remainder[i, j], i, j) for i in range(matrix.shape[0]) for j in range(matrix.shape[1])), reverse=True)
    for _, i, j in order:
        while need_rows[i] > 0 and need_cols[j] > 0 and (floor[i, j] < caps[i, j]):
            floor[i, j] += 1
            need_rows[i] -= 1
            need_cols[j] -= 1
    if need_rows.sum() or need_cols.sum():
        raise ValueError(f'Не удалось разложить квоты по клеткам: пула не хватает.\nнедобор по строкам:   {need_rows.tolist()}\nнедобор по столбцам: {need_cols.tolist()}')
    return floor

def integer_margins(shares: np.ndarray, total: int) -> np.ndarray:
    raw = shares / shares.sum() * total
    base = np.floor(raw).astype(int)
    for i in np.argsort(-(raw - base))[:total - int(base.sum())]:
        base[i] += 1
    return base.astype(float)
caps = pool_table.values.astype(int)
if ALLOCATION_MODE == 'balanced':
    row_targets = integer_margins(np.ones(len(STRATA[STRATA_ROW])), N_CASES)
    col_targets = integer_margins(np.ones(len(STRATA[STRATA_COL])), N_CASES)
elif ALLOCATION_MODE == 'proportional':
    row_targets = integer_margins(caps.sum(axis=1).astype(float), N_CASES)
    col_targets = integer_margins(caps.sum(axis=0).astype(float), N_CASES)
else:
    raise ValueError(f'Неизвестный ALLOCATION_MODE: {ALLOCATION_MODE!r}')
print('Целевые краевые суммы (режим:', ALLOCATION_MODE, ')')
print(f'  {STRATA_ROW:<10} ' + ', '.join((f'{lv}: {int(v)}' for lv, v in zip(STRATA[STRATA_ROW], row_targets))))
print(f'  {STRATA_COL:<10} ' + ', '.join((f'{lv}: {int(v)}' for lv, v in zip(STRATA[STRATA_COL], col_targets))))
print()
fractional = ipf(caps.astype(float), row_targets, col_targets)
print('IPF, дробное решение:')
print(pd.DataFrame(np.round(fractional, 2), index=STRATA[STRATA_ROW], columns=STRATA[STRATA_COL]).to_string())
quota = round_preserving_margins(fractional, row_targets, col_targets, caps)
quota_table = pd.DataFrame(quota, index=STRATA[STRATA_ROW], columns=STRATA[STRATA_COL])
print()
print('Квоты по клеткам (целые):')
print(quota_table.to_string())
print(f'  по строкам:  {quota.sum(axis=1).tolist()}')
print(f'  по столбцам: {quota.sum(axis=0).tolist()}')
print(f'  всего: {int(quota.sum())} (нужно {N_CASES})')
assert int(quota.sum()) == N_CASES, 'квоты не складываются в N_CASES'
assert (quota <= caps).all(), 'квота превышает запас пула в какой-то клетке'
print()
print('Остаток пула после отбора (сколько кейсов в клетке не использовано):')
print(pd.DataFrame(caps - quota, index=STRATA[STRATA_ROW], columns=STRATA[STRATA_COL]).to_string())


Целевые краевые суммы (режим: balanced )
  Genealogy  AI: 20, checker: 20, msd: 20
  Type       Requires emergent care: 20, Requires non-emergent care: 20, Self-care appropriate: 20

IPF, дробное решение:


         Requires emergent care  Requires non-emergent care  Self-care appropriate
AI                         5.16                        4.39                  10.45
checker                    4.98                        5.48                   9.55
msd                        9.87                       10.13                   0.00

Квоты по клеткам (целые):
         Requires emergent care  Requires non-emergent care  Self-care appropriate
AI                            5                           4                     11
checker                       6                           5                      9
msd                           9                          11                      0
  по строкам:  [20, 20, 20]
  по столбцам: [20, 20, 20]
  всего: 60 (нужно 60)

Остаток пула после отбора (сколько кейсов в клетке не использовано):
         Requires emergent care  Requires non-emergent care  Self-care appropriate
AI                           47                          42                  

In [7]:
position_in_corpus = {key: i + 1 for i, key in enumerate(ordered_keys)}
selection_rng = np.random.default_rng(SELECTION_SEED)
selected_keys: List[str] = []
for row_index, row_level in enumerate(STRATA[STRATA_ROW]):
    for col_index, col_level in enumerate(STRATA[STRATA_COL]):
        take = int(quota[row_index, col_index])
        if take == 0:
            continue
        members = [k for k in eligible if corpus.at[k, STRATA_ROW] == row_level and corpus.at[k, STRATA_COL] == col_level]
        chosen = selection_rng.choice(len(members), size=take, replace=False)
        selected_keys.extend((members[i] for i in sorted(chosen)))
selected_keys = sorted(selected_keys, key=lambda k: position_in_corpus[k])
print(f'Отобрано кейсов: {len(selected_keys)} (уникальных: {len(set(selected_keys))})')
assert len(selected_keys) == N_CASES == len(set(selected_keys))
selection = pd.DataFrame({'n': range(1, len(selected_keys) + 1), 'case_id': [corpus.at[k, 'case_id'] for k in selected_keys], 'corpus_pos': [position_in_corpus[k] for k in selected_keys], STRATA_ROW: [corpus.at[k, STRATA_ROW] for k in selected_keys], STRATA_COL: [corpus.at[k, STRATA_COL] for k in selected_keys], 'specialty': [corpus.at[k, 'Sub Clinical Specialties'] for k in selected_keys], 'age_group': [corpus.at[k, 'Age Group'] for k in selected_keys], 'diagnosis': [str(corpus.at[k, 'Diagnosis_vulgar'])[:42] for k in selected_keys]})
for name in SOURCE_ORDER:
    selection[f'len_{name}'] = complete_tables[name]['text'].reindex(selected_keys).str.len().values
with pd.option_context('display.max_rows', None, 'display.width', 250, 'display.max_colwidth', 44):
    print(selection.to_string(index=False))


Отобрано кейсов: 60 (уникальных: 60)


 n    case_id  corpus_pos Genealogy                       Type                 specialty          age_group                                  diagnosis  len_delphi  len_deepseekv4  len_gpt  len_qwen
 1 951a7c6bbb           6       msd Requires non-emergent care               Pulmonology middle age (45-60)                                     Asthma        3254            2622     2658      2010
 2 6f3d2f1324          15       msd     Requires emergent care                   Urology  young age (25-44)          Acute pyelonephritis in pregnancy        3681            2869     2395      2638
 3 7dad308ea3          23       msd Requires non-emergent care                Cardiology middle age (45-60)                        Atrial fibrillation        3764            3402     2467      2650
 4 2c86ec530c          29        AI      Self-care appropriate        Infectious Disease     children (<15)             Hand-foot-mouth disease (mild)        3001            2178     1900      1895
 5 c53edcd

In [8]:
merged = corpus.loc[selected_keys].reset_index()[CASE_COLUMNS].copy()
for name in SOURCE_ORDER:
    merged[f'protocol__{name}'] = complete_tables[name]['text'].reindex(selected_keys).values
named_protocol_columns = [f'protocol__{n}' for n in SOURCE_ORDER]
short = merged[named_protocol_columns].astype(str).apply(lambda s: (s.str.len() < MIN_PROTOCOL_CHARS).sum())
print('Пустых/коротких протоколов после мерджа:')
print(short.to_string())
if int(short.sum()) > 0:
    raise ValueError('В наборе оказался неполный протокол - проверьте шаг 2.')
leftover = int(merged[named_protocol_columns].astype(str).apply(lambda s: s.str.contains('(?im)^\\s*Protocol Author\\s*:', regex=True)).sum().sum())
print(f"\nОсталось хвостов 'Protocol Author': {leftover}")
print(f'Мердж: {merged.shape[0]} строк x {merged.shape[1]} колонок')
print('Колонки:', list(merged.columns))


Пустых/коротких протоколов после мерджа:


protocol__delphi        0
protocol__deepseekv4    0
protocol__gpt           0
protocol__qwen          0

Осталось хвостов 'Protocol Author': 0
Мердж: 60 строк x 13 колонок
Колонки: ['case_id', 'Vignette', 'case_text', 'Criterion', 'Diagnosis_vulgar', 'Type', 'Genealogy', 'Sub Clinical Specialties', 'Age Group', 'protocol__delphi', 'protocol__deepseekv4', 'protocol__gpt', 'protocol__qwen']


In [9]:
rng = np.random.default_rng(PERMUTATION_SEED)
anonymous_columns = [f'protocol_{i + 1}' for i in range(len(SOURCE_ORDER))]
source_columns = [f'source_of_{i + 1}' for i in range(len(SOURCE_ORDER))]
anonymous_values = {column: [] for column in anonymous_columns}
source_values = {column: [] for column in source_columns}
perm_keys: List[str] = []
for _, row in merged.iterrows():
    positions = rng.permutation(len(SOURCE_ORDER))
    slots: List[Optional[str]] = [None] * len(SOURCE_ORDER)
    owners: List[Optional[str]] = [None] * len(SOURCE_ORDER)
    for source_index, target_position in enumerate(positions):
        name = SOURCE_ORDER[source_index]
        slots[target_position] = row[f'protocol__{name}']
        owners[target_position] = name
    for i in range(len(SOURCE_ORDER)):
        anonymous_values[anonymous_columns[i]].append(slots[i])
        source_values[source_columns[i]].append(owners[i])
    perm_keys.append(','.join((str(p + 1) for p in positions)))
blind = merged[CASE_COLUMNS].copy()
for column in anonymous_columns:
    blind[column] = anonymous_values[column]
keyed = blind.copy()
keyed['perm_key'] = perm_keys
for column in source_columns:
    keyed[column] = source_values[column]
print('Распределение позиций по источникам (должно быть примерно равномерным):')
distribution = pd.DataFrame({name: [sum((1 for row in keyed[source_columns].values if row[i] == name)) for i in range(len(SOURCE_ORDER))] for name in SOURCE_ORDER}, index=[f'позиция {i + 1}' for i in range(len(SOURCE_ORDER))])
print(distribution.to_string())


Распределение позиций по источникам (должно быть примерно равномерным):
           delphi  deepseekv4  gpt  qwen
позиция 1      14          22   13    11
позиция 2      21          10   13    16
позиция 3       9          15   17    19
позиция 4      16          13   17    14


In [10]:
restored_ok = 0
for row_index in range(len(keyed)):
    row = keyed.iloc[row_index]
    for position in range(len(SOURCE_ORDER)):
        owner = row[f'source_of_{position + 1}']
        original = merged.iloc[row_index][f'protocol__{owner}']
        if row[f'protocol_{position + 1}'] == original:
            restored_ok += 1
print(f'1. Обратимость: {restored_ok}/{len(keyed) * len(SOURCE_ORDER)} протоколов восстановлены к своему источнику')
replay_rng = np.random.default_rng(PERMUTATION_SEED)
replayed = [','.join((str(p + 1) for p in replay_rng.permutation(len(SOURCE_ORDER)))) for _ in range(len(merged))]
print(f"2. Воспроизведение перестановки из seed={PERMUTATION_SEED}: {('совпало полностью' if replayed == perm_keys else 'РАСХОЖДЕНИЕ')}")
leak_columns = [c for c in blind.columns if 'source' in c.lower() or 'perm' in c.lower() or any((name in c for name in SOURCE_ORDER))]
print(f"3. Ключевые колонки в слепом файле: {(leak_columns if leak_columns else 'нет')}")
markers = ['Protocol Author', 'delphi_consensus', 'gpt120b', 'qwen_default', 'deepseekv4', 'gpt-oss-120b', 'qwen3-235b', 'deepseek-v4']
print('4. Маркеры источника в тексте протоколов:')
for marker in markers:
    hits = int(sum((blind[c].astype(str).str.contains(marker, case=False, regex=False).sum() for c in anonymous_columns)))
    print(f'   {marker:<20} {hits}')
achieved = pd.crosstab(blind[STRATA_ROW], blind[STRATA_COL]).reindex(index=STRATA[STRATA_ROW], columns=STRATA[STRATA_COL], fill_value=0)
print('5. Фактическое распределение страт:')
print(achieved.to_string())
print(f'   совпадает с квотами: {bool((achieved.values == quota).all())}')


1. Обратимость: 240/240 протоколов восстановлены к своему источнику
2. Воспроизведение перестановки из seed=20260824: совпало полностью
3. Ключевые колонки в слепом файле: нет
4. Маркеры источника в тексте протоколов:
   Protocol Author      0
   delphi_consensus     0
   gpt120b              0
   qwen_default         0
   deepseekv4           0
   gpt-oss-120b         0
   qwen3-235b           0
   deepseek-v4          0
5. Фактическое распределение страт:
Type       Requires emergent care  Requires non-emergent care  Self-care appropriate
Genealogy                                                                           
AI                              5                           4                     11
checker                         6                           5                      9
msd                             9                          11                      0
   совпадает с квотами: True


In [11]:
length_by_source = {name: int(merged[f'protocol__{name}'].astype(str).str.len().median()) for name in SOURCE_ORDER}
length_by_position = {column: int(blind[column].astype(str).str.len().median()) for column in anonymous_columns}
print('Медианная длина протокола, знаков:')
print('  по источнику :', length_by_source)
print('  по позиции   :', length_by_position)
longest_is_delphi = sum((1 for _, row in keyed.iterrows() if row[f'source_of_{int(np.argmax([len(str(row[c])) for c in anonymous_columns])) + 1}'] == 'delphi'))
print(f'\nСамый длинный протокол в строке принадлежит delphi: {longest_is_delphi}/{len(keyed)} ({100.0 * longest_is_delphi / len(keyed):.0f}%)')
print('Это верхняя оценка того, насколько легко снять слепоту по одной лишь длине.')
print()
print('Смещение набора относительно корпуса:')
report_rows = []
for column, levels in STRATA.items():
    corpus_counts = corpus[column].value_counts()
    for level in levels:
        in_corpus = int(corpus_counts.get(level, 0))
        in_set = int((blind[column] == level).sum())
        share_corpus = in_corpus / len(corpus)
        share_set = in_set / len(blind)
        report_rows.append({'разметка': column, 'уровень': level, 'в корпусе': in_corpus, 'доля корпуса': round(share_corpus, 3), 'в наборе': in_set, 'доля набора': round(share_set, 3), 'вес на корпус': round(share_corpus / share_set, 3) if share_set else np.nan})
bias = pd.DataFrame(report_rows)
print(bias.to_string(index=False))
print()
print('Побочные разметки (не балансировались, смотрим только на покрытие):')
for column in ['Sub Clinical Specialties', 'Age Group']:
    counts = blind[column].value_counts()
    print(f'  {column}: {len(counts)} уровней, крупнейший {counts.index[0]!r} = {int(counts.iloc[0])}/{len(blind)}')


Медианная длина протокола, знаков:
  по источнику : {'delphi': 3602, 'deepseekv4': 2248, 'gpt': 2030, 'qwen': 2044}
  по позиции   : {'protocol_1': 2504, 'protocol_2': 2458, 'protocol_3': 2316, 'protocol_4': 2304}



Самый длинный протокол в строке принадлежит delphi: 58/60 (97%)
Это верхняя оценка того, насколько легко снять слепоту по одной лишь длине.

Смещение набора относительно корпуса:
 разметка                    уровень  в корпусе  доля корпуса  в наборе  доля набора  вес на корпус
Genealogy                         AI        151         0.659        20        0.333          1.978
Genealogy                    checker         43         0.188        20        0.333          0.563
Genealogy                        msd         35         0.153        20        0.333          0.459
     Type     Requires emergent care         84         0.367        20        0.333          1.100
     Type Requires non-emergent care         81         0.354        20        0.333          1.061
     Type      Self-care appropriate         64         0.279        20        0.333          0.838

Побочные разметки (не балансировались, смотрим только на покрытие):
  Sub Clinical Specialties: 16 уровней, крупнейший 

In [12]:
blind.to_csv(OUT_BLIND, sep=';', index=False, encoding='utf-8-sig')
keyed.to_csv(OUT_KEYED, sep=';', index=False, encoding='utf-8-sig')
manifest = selection.copy()
manifest['delphi_run'] = DELPHI_DIR
manifest['allocation_mode'] = ALLOCATION_MODE
manifest['selection_seed'] = SELECTION_SEED
weight = {(r['разметка'], r['уровень']): r['вес на корпус'] for _, r in bias.iterrows()}
manifest[f'weight_{STRATA_ROW}'] = [weight[STRATA_ROW, v] for v in manifest[STRATA_ROW]]
manifest[f'weight_{STRATA_COL}'] = [weight[STRATA_COL, v] for v in manifest[STRATA_COL]]
manifest.to_csv(OUT_SELECTION, sep=';', index=False, encoding='utf-8-sig')
print(f'Слепой файл  : {OUT_BLIND}')
print(f'  {blind.shape[0]} строк x {blind.shape[1]} колонок')
print(f'  {list(blind.columns)}')
print()
print(f'Файл с ключом: {OUT_KEYED}')
print(f'  {keyed.shape[0]} строк x {keyed.shape[1]} колонок')
print()
print(f'Манифест     : {OUT_SELECTION}')
print(f'  {manifest.shape[0]} строк x {manifest.shape[1]} колонок')
print()
print(f'Дельфийский прогон: {DELPHI_DIR}')
print(f'Seed отбора кейсов (можно публиковать): {SELECTION_SEED}')
print(f'Seed перестановки (хранить отдельно от слепого файла): {PERMUTATION_SEED}')
print()
print('Пример расшифровки первой строки:')
first = keyed.iloc[0]
print(f"  case_id = {first['case_id']}  perm_key = {first['perm_key']}")
print(f'  (порядок источников для perm_key: {SOURCE_ORDER})')
for position in range(len(SOURCE_ORDER)):
    print(f"  protocol_{position + 1} -> {first[f'source_of_{position + 1}']}")


Слепой файл  : .\evaluations\human_and_model_evaluations/human_and_model_validation/blind_protocols_60.csv
  60 строк x 13 колонок
  ['case_id', 'Vignette', 'case_text', 'Criterion', 'Diagnosis_vulgar', 'Type', 'Genealogy', 'Sub Clinical Specialties', 'Age Group', 'protocol_1', 'protocol_2', 'protocol_3', 'protocol_4']

Файл с ключом: .\evaluations\human_and_model_evaluations/human_and_model_validation/blind_protocols_60_key.csv
  60 строк x 18 колонок

Манифест     : .\evaluations\human_and_model_evaluations/human_and_model_validation/blind_protocols_60_selection.csv
  60 строк x 17 колонок

Дельфийский прогон: heterogeneous_gpt120_organizer_complete
Seed отбора кейсов (можно публиковать): 20260831
Seed перестановки (хранить отдельно от слепого файла): 20260824

Пример расшифровки первой строки:
  case_id = 951a7c6bbb  perm_key = 2,3,4,1
  (порядок источников для perm_key: ['delphi', 'deepseekv4', 'gpt', 'qwen'])
  protocol_1 -> qwen
  protocol_2 -> delphi
  protocol_3 -> deepseekv4
 